# Notebook 02 — Fine-tune Wav2Vec2 for Dementia Classification

Loads `train_dm.csv` / `valid_dm.csv` produced by notebook 01 and fine-tunes  
`facebook/wav2vec2-xls-r-300m` for binary speech classification.

In [ ]:
%%capture
!pip install transformers datasets evaluate
!sudo apt-get install -y git-lfs
!git lfs install

In [ ]:
import os
import sys
import numpy as np
import torch
import torch.nn as nn
import torchaudio

from random import randint
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple, Union
from pathlib import Path

from datasets import load_dataset
from transformers import (
    AutoConfig,
    EvalPrediction,
    Trainer,
    TrainingArguments,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Model,
    Wav2Vec2PreTrainedModel,
)
from transformers.file_utils import ModelOutput
from huggingface_hub import notebook_login

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if 'failed' in gpu_info:
    print('No GPU detected. Go to Runtime > Change runtime type and select a GPU.')
else:
    print(gpu_info)

In [ ]:
notebook_login()

## Configuration

In [ ]:
data_path    = Path('/content/drive/MyDrive/CS7357/Project/data')
repo_name    = 'wav2vec2-large-xls-r-300m-dm32'
model_name   = 'facebook/wav2vec2-xls-r-300m'
pooling_mode = 'mean'
input_col    = 'path'
output_col   = 'label'
audio_len    = 32   # seconds to randomly subsample per clip

# Set to True when rerunning after changes to notebook 01 (SpecAugment params,
# augmentation copies, dataset split, etc.). Leave False to reuse cached
# preprocessed data and skip straight to training.
CLEAR_CACHE = False

## 1. Load Dataset

In [ ]:
data_files = {
    'train': str(data_path / 'train_dm.csv'),
    'valid': str(data_path / 'valid_dm.csv'),
}
dataset    = load_dataset('csv', data_files=data_files, delimiter='\t')
train_data = dataset['train']
valid_data = dataset['valid']

label_list  = sorted(train_data.unique(output_col))
num_classes = len(label_list)

print(f'Train : {len(train_data)} samples')
print(f'Valid : {len(valid_data)} samples')
print(f'Classes ({num_classes}): {label_list}')

# Compute inverse-frequency class weights to penalise the minority class
from collections import Counter
label_counts  = Counter(train_data[output_col])
total_samples = sum(label_counts.values())
class_weights = torch.tensor(
    [total_samples / (num_classes * label_counts[l]) for l in label_list],
    dtype=torch.float,
)
print('Class weights:', {l: f'{w:.3f}' for l, w in zip(label_list, class_weights)})

## 2. Model Config & Feature Extractor

In [ ]:
config = AutoConfig.from_pretrained(
    model_name,
    num_labels=num_classes,
    label2id={label: i for i, label in enumerate(label_list)},
    id2label={i: label for i, label in enumerate(label_list)},
    finetuning_task='wav2vec2_clf',
)
setattr(config, 'pooling_mode', pooling_mode)

feature_extractor    = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
target_sampling_rate = feature_extractor.sampling_rate
print(f'Target sampling rate: {target_sampling_rate}')

## 3. Preprocessing

In [ ]:
import shutil
def random_subsample(wav: np.ndarray, max_length: float, sample_rate: int = 16000) -> np.ndarray:
    """Randomly crop a chunk of `max_length` seconds from the waveform."""
    sample_length = int(round(sample_rate * max_length))
    if len(wav) <= sample_length:
        return wav
    offset = randint(0, len(wav) - sample_length - 1)
    return wav[offset : offset + sample_length]


def speech_to_array(path: str) -> np.ndarray:
    speech, sr = torchaudio.load(path)
    speech = torchaudio.transforms.Resample(sr, target_sampling_rate)(speech)[0].numpy()
    return random_subsample(speech, max_length=audio_len)


def label_to_id(label: str, label_list: list) -> int:
    return label_list.index(label) if label in label_list else -1


def preprocess_fn(examples):
    speech_list = [speech_to_array(p) for p in examples[input_col]]
    target_list = [label_to_id(l, label_list) for l in examples[output_col]]
    input_values = [
        feature_extractor(s, sampling_rate=target_sampling_rate).input_values[0]
        for s in speech_list
    ]
    return {'input_values': input_values, 'labels': list(target_list)}


from datasets import load_from_disk

cache_train_path = str(data_path / 'cache_train')
cache_valid_path = str(data_path / 'cache_valid')

if CLEAR_CACHE:
    for p in [cache_train_path, cache_valid_path]:
        if Path(p).exists():
            shutil.rmtree(p)
            print(f'Cleared cache: {p}')

if not CLEAR_CACHE and Path(cache_train_path).exists() and Path(cache_valid_path).exists():
    train_data = load_from_disk(cache_train_path)
    valid_data = load_from_disk(cache_valid_path)
    print('Loaded preprocessed data from cache.')
else:
    train_data = train_data.map(preprocess_fn, batch_size=8, batched=True, num_proc=1)
    valid_data = valid_data.map(preprocess_fn, batch_size=8, batched=True, num_proc=1)
    train_data.save_to_disk(cache_train_path)
    valid_data.save_to_disk(cache_valid_path)
    print('Preprocessed and saved cache to Drive.')

## 4. Model Architecture

In [ ]:
@dataclass
class SpeechClassifierOutput(ModelOutput):
    loss:          Optional[torch.FloatTensor]            = None
    logits:        torch.FloatTensor                      = None
    hidden_states: Optional[Tuple[torch.FloatTensor]]    = None
    attentions:    Optional[Tuple[torch.FloatTensor]]    = None


class Wav2Vec2ClassificationHead(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.dense    = nn.Linear(config.hidden_size, config.hidden_size)
        self.dropout  = nn.Dropout(config.final_dropout)
        self.out_proj = nn.Linear(config.hidden_size, config.num_labels)

    def forward(self, x, **kwargs):
        x = self.dropout(x)
        x = self.dense(x)
        x = torch.tanh(x)
        x = self.dropout(x)
        x = self.out_proj(x)
        return x


class Wav2Vec2ForSpeechClassification(Wav2Vec2PreTrainedModel):
    all_tied_weights_keys = {}

    def __init__(self, config):
        super().__init__(config)
        self.num_labels  = config.num_labels
        self.pooling_mode = config.pooling_mode
        self.wav2vec2    = Wav2Vec2Model(config)
        self.classifier  = Wav2Vec2ClassificationHead(config)
        self.init_weights()

    def freeze_feature_extractor(self):
        self.wav2vec2.feature_extractor._freeze_parameters()

    def _pool(self, hidden_states):
        if self.pooling_mode == 'mean':
            return hidden_states.mean(dim=1)
        elif self.pooling_mode == 'max':
            return hidden_states.max(dim=1)[0]
        elif self.pooling_mode == 'sum':
            return hidden_states.sum(dim=1)
        raise ValueError(f'Unknown pooling mode: {self.pooling_mode}')

    def forward(self, input_values, attention_mask=None, output_attentions=None,
                output_hidden_states=None, return_dict=None, labels=None):
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict
        outputs = self.wav2vec2(
            input_values,
            attention_mask=attention_mask,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )
        logits = self.classifier(self._pool(outputs[0]))

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits.view(-1, self.num_labels), labels.view(-1))

        if not return_dict:
            return ((loss, logits) + outputs[2:]) if loss is not None else (logits,) + outputs[2:]

        return SpeechClassifierOutput(
            loss=loss, logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

## 5. Data Collator & Metrics

In [ ]:
@dataclass
class DataCollatorCTCWithPadding:
    feature_extractor: Wav2Vec2FeatureExtractor
    padding:           Union[bool, str] = True
    max_length:        Optional[int]    = None
    pad_to_multiple_of: Optional[int]  = None

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        input_features = [{'input_values': f['input_values']} for f in features]
        label_features = [f['labels'] for f in features]
        d_type = torch.long if isinstance(label_features[0], int) else torch.float
        batch  = self.feature_extractor.pad(
            input_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors='pt',
        )
        batch['labels'] = torch.tensor(label_features, dtype=d_type)
        return batch


def compute_metrics(p: EvalPrediction) -> Dict:
    preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    preds = np.argmax(preds, axis=1)
    return {'accuracy': (preds == p.label_ids).astype(np.float32).mean().item()}


data_collator = DataCollatorCTCWithPadding(feature_extractor=feature_extractor, padding=True)


class WeightedTrainer(Trainer):
    """Trainer that applies per-class loss weighting."""
    def __init__(self, class_weights: torch.Tensor, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        loss    = nn.CrossEntropyLoss(
            weight=self.class_weights.to(outputs.logits.device)
        )(outputs.logits.view(-1, model.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

## 6. Load Model

In [ ]:
# Transformers inspects __main__.__file__ when loading a custom model class defined
# in a notebook; this workaround prevents a FileNotFoundError.
if not hasattr(sys.modules['__main__'], '__file__'):
    with open('notebook_source.py', 'w') as f:
        f.write('')
    sys.modules['__main__'].__file__ = os.path.abspath('notebook_source.py')

model = Wav2Vec2ForSpeechClassification.from_pretrained(model_name, config=config)
model.freeze_feature_extractor()

## 7. Train

In [ ]:
training_args = TrainingArguments(
    output_dir=repo_name,
    group_by_length=False,           # requires input_ids; we use input_values
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    eval_strategy='steps',
    gradient_checkpointing=True,
    num_train_epochs=22,
    save_steps=110,
    eval_steps=34,
    logging_steps=110,
    learning_rate=1e-4,
    save_total_limit=2,
    fp16=True,
    push_to_hub=True,
)

trainer = WeightedTrainer(
    class_weights=class_weights,
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_data,
    eval_dataset=valid_data,
)

trainer.train()

## 8. Push to Hub

In [ ]:
trainer.push_to_hub(repo_name)
feature_extractor.push_to_hub(repo_name)
print('Model and feature extractor pushed to hub.')